# 模型的评估标准

In [6]:
import numpy as np
import torch
from PIL import Image

APPLE_PATH = "datas/LR_Fruits/apples.png"
BANANA_PATH = "datas/LR_Fruits/bananas.png"

torch.set_num_threads(1)
device = ""
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

def load_doodles(path):
    # 大图由 100 行 15 列、共 1500 张 28×28 的涂鸦拼成，先把它切回一张张涂鸦
    grid = np.array(Image.open(path))
    print(grid.shape)
    return grid.reshape(100, 28, 15, 28).swapaxes(1, 2).reshape(1500, 28, 28)
# 1500张图片，每张大小 28*28
apples = load_doodles(APPLE_PATH)
bananas = load_doodles(BANANA_PATH)

# 将两张图片拼接起来
images_concatenate = np.concatenate([apples, bananas], axis = 0)

# 归一化，这里的最大值其实就是255，最小值0
mins = images_concatenate.min(axis=(1, 2), keepdims=True)
maxs = images_concatenate.max(axis=(1, 2), keepdims=True)
# 归一化 + 反相
images_normalized = 1 - (images_concatenate - mins) / (maxs - mins)

# 变成 tensor
images = torch.tensor(images_normalized, dtype=torch.float32, device=device)

# 将图片展开成一维
img_input = images.reshape(images.shape[0], images.shape[1] * images.shape[2])

# 标签
labels=torch.cat([torch.ones(apples.shape[0],device=device), torch.zeros(bananas.shape[0], device=device)])

# 生成一个 0～2999 的随机序列，打乱数据
order = torch.randperm(3000, generator=torch.Generator().manual_seed(42)).to(device)

train_ids = order[:1800]
test_id = order[1800: 2400]
val_ids = order[2400:]

x_train, y_train = img_input[train_ids], labels[train_ids]
x_test, y_test = img_input[test_id], labels[test_id]
x_val, y_test = img_input[val_ids], labels[val_ids]

# 参数初始化（权重，偏置，损失函数，优化器）
weights = torch.zeros(784, requires_grad=True, device=device)
bias = torch.zeros(1, requires_grad=True,device=device)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD([weights, bias], lr=0.2)

num_steps=500
train_losses, val_losses = [], []
for step in range(num_steps):
    # 计算结果
    logits = x_train @ weights + bias

    # 误差
    train_loss = loss_fn(logits, y_train)
    with torch.no_grad():
        val_logits = x_val @ weights + bias
        val_loss = loss_fn(val_logits, y_test)

    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    if step % 100 == 0:
        print(f"训练集误差{train_loss}, 验证集误差{val_loss}")

    # 梯度下降
    optimizer.zero_grad()# 清空梯度
    train_loss.backward()# 计算梯度
    optimizer.step()# 梯度下降

(2800, 420)
(2800, 420)
训练集误差0.6931471824645996, 验证集误差0.6931471824645996
训练集误差0.10740480571985245, 验证集误差0.12239141017198563
训练集误差0.09232541918754578, 验证集误差0.10951927304267883
训练集误差0.0846462994813919, 验证集误差0.10527660697698593
训练集误差0.0792500451207161, 验证集误差0.10352946072816849


## 召回率与精确率——结合混淆矩阵
> 1. recall——看看是否找全
> 2. precision——看看是否找准
> 3. F1——同时看 precision 和 recall

In [41]:
with torch.no_grad():
    probabilities = torch.sigmoid(x_test @ weights + bias)
# 苹果是 1，香蕉是 0
def evaluate(threshold):
    y_test_predict = (probabilities >= threshold).float()
    tp = ((y_test_predict == 1) & (y_test == 1)).sum()
    fp = ((y_test_predict == 1) & (y_test == 0)).sum()
    fn = ((y_test_predict == 0) & (y_test == 1)).sum()

    _precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    _recall = tp / (tp + fn)
    _F1 = 2 * _precision * _recall / (_precision + _recall)
    return float(_precision), float(_recall),float(_F1)
for i in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    precision, recall, F1 = evaluate(i)
    print(f"threshold:{i}\t recall:{recall:.3f}\t precision:{precision:.3f} \tF1 score:{F1:.3f}")

threshold:0.1	 recall:0.980	 precision:0.869 	F1 score:0.921
threshold:0.2	 recall:0.977	 precision:0.893 	F1 score:0.933
threshold:0.3	 recall:0.970	 precision:0.927 	F1 score:0.948
threshold:0.4	 recall:0.970	 precision:0.929 	F1 score:0.949
threshold:0.5	 recall:0.967	 precision:0.938 	F1 score:0.952
threshold:0.6	 recall:0.957	 precision:0.947 	F1 score:0.952
threshold:0.7	 recall:0.946	 precision:0.959 	F1 score:0.953
threshold:0.8	 recall:0.936	 precision:0.966 	F1 score:0.951
threshold:0.9	 recall:0.893	 precision:0.982 	F1 score:0.935
